In [18]:
from ollama import chat
import pandas as pd
import json
import base64
import numpy as np
from collections import Counter
from pyspark.sql import SparkSession

In [19]:
with open("text-to-plot/combined/combined.json", "r") as f:
    combined = json.load(f)
with open("text-to-plot/combined/combined-without-type.json", "r") as f:
    combined_without_type = json.load(f)
with open("text-to-plot/combined/combined-with-type.json", "r") as f:
    combined_with_type = json.load(f)
with open("text-to-plot/combined/combined-code.json", "r") as f:
    combined_code = json.load(f)
with open("text-to-plot/combined/combined-golden.json", "r") as f:
    combined_golden = json.load(f)

In [20]:
grouped_golden = {}
for c in combined_golden:
    grouped_golden[c['uuid']] = c['plot_json']['data']

In [21]:
def generate(system_message: str, user_message: str, model: str):
    return chat(
        model=model, 
        messages =[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}],
        options={
            "temperature": 0
        }
    )

In [22]:
def save_json(file_name: str, data):
    with open(file_name, "w") as f:
        json.dump(data, f, indent=4) 

In [23]:
def decode_bdata(entry):
    raw = base64.b64decode(entry["bdata"])
    arr = np.frombuffer(raw, dtype=np.dtype(entry["dtype"]))
    return arr

In [24]:
def get_chart_code(content: str):
    return content.split("</chart-code>")[0].split("<chart-code>")[1]

In [25]:
def get_thinking(content: str):
    return content.split("</thinking>")[0].split("<thinking>")[1]

In [26]:
def compare_results(result, uuid):
    gt = grouped_golden[uuid]
    type_hit = 0
    hit = 0.0
    miss = 0.0

    for key in gt.keys():
        if key == "type":
            type_hit += gt[key] == result[key]
        if type(gt[key]) == list:
            values = result[key]
            if 'dtype' in values:
                values = decode_bdata(values)
            hit += Counter(values) == Counter(gt[key])
            miss += Counter(values) != Counter(gt[key])
        else:
            hit += result[key] == gt[key]
            miss += result[key] != gt[key]

    return {
        "hit": type_hit,
        "score": hit / (hit + miss)
    }

In [27]:
def load_examples(dataset_path):
    filename = dataset_path.split("/")[-1]

    spark = SparkSession.builder.appName("CSVExample").getOrCreate()

    df = spark.read.csv(f"text-to-plot/datasets/{filename}", header=True, inferSchema=True)

    rows = df.take(5)
    rows_as_strings = [str(row) for row in rows]
    return {
        "df": df,
        "examples": "\n".join(rows_as_strings)
    }

### WithoutChartType

In [28]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetIndex` is not supported. for index 2
Exception 'domain' for index 3
Finished index 4
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 5
Exception 'domain' for index 6
Exception 'domain' for index 7
Exception Column.contains() got an unexpected keyword argument 'regex' for index 8
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 9
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 10
Exception name 'col' is not defined for index 11
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 12
Finished index 13
Exception 'Column' object is not callable for index 14
Exception bar() got an unexpected keyword argument 'data' for index 15
Exception 'DataFrame' object does not support item assignment for index 16
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetRo

{"ts": "2025-09-14 17:18:43.679", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 10 in cell [29]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o541.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 10 in cell [29]\n\r\n\tat org.apache.spark.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 10 in cell [29]
 for index 20


{"ts": "2025-09-14 17:18:46.508", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 10 in cell [29]", "line": "", "fragment": "isin", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o578.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"isin\" was called from\nline 10 in cell [29]\n\r\n\tat org.apache.spark.sql.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"isin" was called from
line 10 in cell [29]
 for index 21
Exception [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/HomePC/Desktop/Docs/Python/NLP/wine.csv. SQLSTATE: 42K03 for index 22
Exception Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['volatile acidity', 'avg(fixed acidity)', 'avg(volatile acidity)', 'avg(citric acid)', 'avg(residual sugar)', 'avg(chlorides)', 'avg(free sulfur dioxide)', 'avg(total sulfur dioxide)', 'avg(density)', 'avg(pH)', 'avg(sulphates)', 'avg(alcohol)', 'avg(quality)'] but received: mean(volatile acidity) for index 23
Exception [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/HomePC/Desktop/Docs/Python/NLP/wines.csv. SQLSTATE: 42K03 

{"ts": "2025-09-14 17:19:16.001", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `City` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider City`, `Provider Id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor99.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o838.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `City` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider City`, `Provider Id`]. SQLSTATE: 42703;\n'Aggregate ['City], ['City, count(1) AS count#2053L]\n+- Filter (Classification#2019 = Alcohol and Drug Use)\n  

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `City` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider City`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['City], ['City, count(1) AS count#2053L]
+- Filter (Classification#2019 = Alcohol and Drug Use)
   +- Relation [City, State#2018,Classification#2019,Definition#2020,DRG#2021,Hospital Referral Region Description#2022,Provider City#2023,Provider Id#2024,Provider Name#2025,Provider State#2026,Provider Street Address#2027,Provider Zip Code#2028,Average Covered Charges #2029,Average Total Payments #2030,Number of Records#2031,Reimbursement Rate#2032,Total Discharges #2033,Total Payment#2034] csv
 for index 32
Finished index 33


{"ts": "2025-09-14 17:19:20.770", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor99.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o885.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;\n'Aggregate ['State], ['State, count(1) AS count#2198L]\n+- Relation [City, State#2164,Classification#2165,De

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['State], ['State, count(1) AS count#2198L]
+- Relation [City, State#2164,Classification#2165,Definition#2166,DRG#2167,Hospital Referral Region Description#2168,Provider City#2169,Provider Id#2170,Provider Name#2171,Provider State#2172,Provider Street Address#2173,Provider Zip Code#2174,Average Covered Charges #2175,Average Total Payments #2176,Number of Records#2177,Reimbursement Rate#2178,Total Discharges #2179,Total Payment#2180] csv
 for index 34
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Description`, `Provider City`, `Provider Id

In [31]:
save_json("text-to-plot/results/llama-32-without.json", results)

#### LLama 3.1 8B

In [32]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Exception 'domain' for index 3
Finished index 4
Exception 'list' object has no attribute 'toPandas' for index 5
Exception 'domain' for index 6
Exception 'domain' for index 7
Exception 'Column' object is not callable for index 8
Finished index 9
Finished index 10
Exception 'domain' for index 11
Finished index 12
Finished index 13
Exception 'domain' for index 14
Exception An error occurred while calling o1776.sum.
: java.lang.ClassCastException: class java.util.HashMap cannot be cast to class java.lang.String (java.util.HashMap and java.lang.String are in module java.base of loader 'bootstrap')
	at scala.collection.immutable.List.map(List.scala:247)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.classic.RelationalGroupedDataset.selectNumericColumns(RelationalGroupedDataset.scala:111)
	at org.apache.spark.sql.RelationalGroupedDataset.aggregateNumericColumns(RelationalGroupedDataset.scala:63)
	at org.apache

{"ts": "2025-09-14 17:22:17.140", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [32]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1844.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 6 in cell [32]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 6 in cell [32]
 for index 18
Exception 'domain' for index 19


{"ts": "2025-09-14 17:22:26.621", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 7 in cell [32]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1888.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 7 in cell [32]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 7 in cell [32]
 for index 20


{"ts": "2025-09-14 17:22:31.229", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [32]", "line": "", "fragment": "__ge__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1920.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__ge__\" was called from\nline 6 in cell [32]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__ge__" was called from
line 6 in cell [32]
 for index 21
Finished index 22
Finished index 23
Finished index 24
Exception 'y' for index 25
Exception 'y' for index 26
Finished index 27
Exception 'y' for index 28
Finished index 29
Exception Cannot accept list of column references or list of columns for both `x` and `y`. for index 30
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Average Total Payments` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Description`, `Provider City`, `Provider Id`, `Provider Name`, `Provider State`, `Provider Street Address`, `Provide

{"ts": "2025-09-14 17:23:26.000", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor99.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2223.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;\n'Aggregate ['State], ['State, count(1) AS count#5358L]\n+- Filter (Classification#5324 = Alcohol and Drug U

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['State], ['State, count(1) AS count#5358L]
+- Filter (Classification#5324 = Alcohol and Drug Use)
   +- Relation [City, State#5323,Classification#5324,Definition#5325,DRG#5326,Hospital Referral Region Description#5327,Provider City#5328,Provider Id#5329,Provider Name#5330,Provider State#5331,Provider Street Address#5332,Provider Zip Code#5333,Average Covered Charges #5334,Average Total Payments #5335,Number of Records#5336,Reimbursement Rate#5337,Total Discharges #5338,Total Payment#5339] csv
 for index 34
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital R

In [34]:
save_json("text-to-plot/results/llama-31-without.json", results)

#### Qwen 3B

In [35]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Exception unexpected indent (<string>, line 1) for index 3
Finished index 4
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `count` cannot be resolved. Did you mean one of the following? [`storenum`, `OPENDATE`, `date_super`, `conversion`, `st`, `county`, `STREETADDR`, `STRCITY`, `STRSTATE`, `ZIPCODE`, `type_store`, `LAT`, `LON`, `MONTH`, `DAY`, `YEAR`]. SQLSTATE: 42703 for index 5
Exception 'count(1)' for index 6
Exception unexpected indent (<string>, line 1) for index 7
Exception 'fig' for index 8
Finished index 9
Finished index 10
Exception unexpected indent (<string>, line 1) for index 11
Finished index 12
Exception All arguments should have the same length. The length of column argument `df[wide_variable_0]` is 1, whereas the length of previously-processed arguments ['y'] is 3 for index 13
Exception name 'col' is not defined for index 14
Finished index 15
Finished index 16
Finished

<string>:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Exception Object of type Interval is not JSON serializable for index 51
Finished index 52
Exception 'y' for index 53


<string>:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Exception Object of type Interval is not JSON serializable for index 54
Finished index 55
26/56


In [36]:
save_json("text-to-plot/results/qwen-without.json", results)

### WithChartType

In [ ]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

In [ ]:
save_json("text-to-plot/results/llama-32-with.json", results)

#### LLama 3.1 8B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

In [ ]:
save_json("text-to-plot/results/llama-31-with.json", results)

#### Qwen 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

In [ ]:
save_json("text-to-plot/results/qwen-with.json", results)

### WithoutChartType + Reasoning

In [11]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Make sure to follow these steps in order and generate nothing else:
1) Give your thinking in the beginning between tags <thinking></thinking>
2) Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<thinking>
To analyze the distribution of national election turnouts in Central/Eastern European countries, I first need to select the turnout data for these countries. Since the data is numerical and continuous, a histogram is suitable to visualize how turnout values are distributed and to identify common ranges and outliers.
</thinking>

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

In [ ]:
save_json("text-to-plot/results/llama-32-reasoning-without-type.json", results)

#### LLama 3.1 8B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

In [ ]:
save_json("text-to-plot/results/llama-31-reasoning-without-type.json", results)

#### Qwen 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")

        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

In [ ]:
save_json("text-to-plot/results/qwen-reasoning-without-type.json", results)